# Log-Normal Regression with E5 Embeddings + Character Length

Training-only reference models for RouterBench inference-cost prediction.

Only the original training partition is used. An internal
group-aware sub-training/sub-validation split is used for early
stopping. Original calibration, validation, and test sets are reserved
for teammates.

In [1]:

# Section 1.1: Load E5 Features and Group-Aware Training Split

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import joblib

PROJECT_ROOT = Path.home() / "Desktop" / "Erdos-Router"

TFIDF_DATA = (
    PROJECT_ROOT
    / "1_data_preparation"
    / "data"
    / "processed"
    / "char_tfidf_50k"
)

E5_DATA = (
    PROJECT_ROOT
    / "1_data_preparation"
    / "data"
    / "processed"
    / "e5_small"
)

# Load only the original training partition
X_tfidf_train = sparse.load_npz(
    TFIDF_DATA / "X_train.npz"
).tocsr()

cost_train = pd.read_pickle(
    TFIDF_DATA / "train_costs.pkl"
)

train_questions = pd.read_pickle(
    TFIDF_DATA / "train_questions.pkl"
)

preprocessing = joblib.load(
    TFIDF_DATA / "preprocessing.joblib"
)

MODELS = list(cost_train.columns)

# Load pretrained E5 embeddings
E5_train = np.load(
    E5_DATA / "E5_train.npy",
    allow_pickle=False
)

# Reuse the existing scaled character-length feature
X_train_length = X_tfidf_train[:, -1].toarray()

# Construct E5 + length feature matrix
X_train_e5 = np.hstack([
    E5_train,
    X_train_length
])

# Reuse the same saved group-aware split
split_path = (
    PROJECT_ROOT
    / "saved_models"
    / "training_splits"
    / "routerbench_grouped_subtrain_subval_seed42.npz"
)

with np.load(split_path) as split:
    subtrain_idx = split["subtrain_idx"]
    subval_idx = split["subval_idx"]

X_subtrain_e5 = X_train_e5[subtrain_idx]
X_subval_e5 = X_train_e5[subval_idx]

cost_subtrain = cost_train.iloc[subtrain_idx].copy()
cost_subval = cost_train.iloc[subval_idx].copy()

# Verify feature dimensions
assert E5_train.shape == (21900, 384)
assert X_train_e5.shape == (21900, 385)
assert X_subtrain_e5.shape == (17520, 385)
assert X_subval_e5.shape == (4380, 385)

# Verify model count and numerical validity
assert len(MODELS) == 11
assert np.isfinite(X_train_e5).all()
assert np.isfinite(cost_train.to_numpy()).all()
assert (cost_train.to_numpy() > 0).all()

# Verify group separation
groups = train_questions["question_group"].to_numpy()

assert set(groups[subtrain_idx]).isdisjoint(
    set(groups[subval_idx])
)

print("E5 embeddings:", E5_train.shape)
print("E5 + length:", X_train_e5.shape)
print("Sub-training:", X_subtrain_e5.shape)
print("Sub-validation:", X_subval_e5.shape)
print("Number of LLMs:", len(MODELS))

print("\nLog-normal E5 training data verified.")


E5 embeddings: (21900, 384)
E5 + length: (21900, 385)
Sub-training: (17520, 385)
Sub-validation: (4380, 385)
Number of LLMs: 11

Log-normal E5 training data verified.


In [2]:

# Section 2.1: Log-Normal Loss and Analytical Gradient

import numpy as np

LOGNORMAL_LAMBDA = 0.1


def lognormal_loss_and_gradient(params, X, y, lam):
    """
    Regularized squared-error regression on log-cost.

    params[0]: intercept
    params[1:]: feature coefficients

    Returns:
        loss: scalar objective
        gradient: NumPy array
    """

    params = np.asarray(params, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    if lam < 0:
        raise ValueError("lambda must be nonnegative.")

    if not np.isfinite(y).all() or (y <= 0).any():
        raise ValueError("Observed costs must be finite and positive.")

    n = X.shape[0]

    intercept = params[0]
    beta = params[1:]

    # Predicted log-cost
    eta = intercept + X @ beta

    # Residuals on the log scale
    residual = eta - np.log(y)

    # Regularized objective
    data_loss = 0.5 * np.mean(residual ** 2)
    penalty = 0.5 * lam * np.dot(beta, beta)

    loss = float(data_loss + penalty)

    # Analytical gradient
    grad_intercept = np.mean(residual)

    grad_beta = (
        np.asarray(X.T @ residual).ravel() / n
        + lam * beta
    )

    gradient = np.concatenate([
        [grad_intercept],
        grad_beta
    ])

    if not np.isfinite(loss) or not np.isfinite(gradient).all():
        raise FloatingPointError(
            "Non-finite log-normal loss or gradient."
        )

    return loss, gradient


print("Log-normal objective and gradient defined.")


Log-normal objective and gradient defined.


In [3]:

# Section 2.2: Numerical Gradient Verification

from scipy.optimize import check_grad

# Small synthetic dataset
X_small = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

y_small = np.array([1.0, 2.0, 3.0])

params_small = np.array([0.2, 0.1, -0.1])

lam = LOGNORMAL_LAMBDA

def objective_only(params):
    return lognormal_loss_and_gradient(
        params, X_small, y_small, lam
    )[0]

def gradient_only(params):
    return lognormal_loss_and_gradient(
        params, X_small, y_small, lam
    )[1]

error = check_grad(
    objective_only,
    gradient_only,
    params_small
)

print("Gradient-check error:", error)

assert error < 1e-6

print("Analytical gradient verified.")


Gradient-check error: 5.556980274123898e-09
Analytical gradient verified.


In [4]:

# Section 3.1: Test L-BFGS Optimization — Log-Normal E5

import numpy as np
from scipy.optimize import minimize

model_name = "meta/llama-2-70b-chat"

# Use only 1,000 internal sub-training observations
X_fit = X_subtrain_e5[:1000]
y_fit = cost_subtrain[model_name].to_numpy()[:1000]

# Initialize intercept at mean log-cost
initial_params = np.zeros(X_fit.shape[1] + 1)
initial_params[0] = np.log(y_fit).mean()

lam = LOGNORMAL_LAMBDA

initial_loss, _ = lognormal_loss_and_gradient(
    initial_params, X_fit, y_fit, lam
)

result = minimize(
    fun=lognormal_loss_and_gradient,
    x0=initial_params,
    args=(X_fit, y_fit, lam),
    method="L-BFGS-B",
    jac=True,
    options={
        "maxiter": 50,
        "ftol": 1e-9
    }
)

print("Initial objective:", initial_loss)
print("Final objective:", result.fun)
print("Iterations:", result.nit)
print("Converged:", result.success)
print("Message:", result.message)

assert np.isfinite(result.fun)
assert result.fun <= initial_loss + 1e-10

print("\nLog-normal E5 optimization verified.")


Initial objective: 0.24894710587482013
Final objective: 0.08559279946815838
Iterations: 14
Converged: True
Message: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL

Log-normal E5 optimization verified.


In [5]:

# Section 3.2: Log-Normal E5 Training with Early Stopping

from scipy.optimize import minimize
import numpy as np
import pandas as pd


def lognormal_log_mse(y_true, predicted_log_cost):
    """Mean squared error on the log-cost scale."""

    y_true = np.asarray(y_true, dtype=np.float64)
    predicted_log_cost = np.asarray(
        predicted_log_cost, dtype=np.float64
    )

    return float(np.mean(
        (np.log(y_true) - predicted_log_cost) ** 2
    ))


def train_lognormal_early_stopping(
    X_fit,
    y_fit,
    X_holdout,
    y_holdout,
    lam=LOGNORMAL_LAMBDA,
    max_iter=200,
    patience=5,
    min_delta=1e-6
):
    """
    Train regularized log-normal regression using L-BFGS-B.

    X_fit/y_fit: internal sub-training partition.
    X_holdout/y_holdout: internal sub-validation partition.

    Retains the lowest observed sub-validation log-MSE.
    """

    y_fit = np.asarray(y_fit, dtype=np.float64)
    y_holdout = np.asarray(y_holdout, dtype=np.float64)

    if X_fit.shape[1] != X_holdout.shape[1]:
        raise ValueError("Feature dimensions must agree.")

    initial_params = np.zeros(X_fit.shape[1] + 1)
    initial_params[0] = np.mean(np.log(y_fit))

    history = []

    best_params = initial_params.copy()
    best_loss = np.inf
    best_iteration = 0

    no_improvement = 0
    iteration = 0

    def record_checkpoint(params, iteration):
        nonlocal best_params, best_loss, best_iteration

        eta_fit = params[0] + X_fit @ params[1:]
        eta_holdout = params[0] + X_holdout @ params[1:]

        fit_log_mse = lognormal_log_mse(y_fit, eta_fit)
        holdout_log_mse = lognormal_log_mse(
            y_holdout, eta_holdout
        )

        history.append({
            "iteration": iteration,
            "subtrain_log_mse": fit_log_mse,
            "subval_log_mse": holdout_log_mse
        })

        if holdout_log_mse < best_loss:
            best_params = params.copy()
            best_loss = holdout_log_mse
            best_iteration = iteration

    # Record initial checkpoint
    record_checkpoint(initial_params, 0)
    previous_best = best_loss

    class EarlyStop(Exception):
        pass

    def callback(params):
        nonlocal iteration, no_improvement, previous_best

        iteration += 1
        record_checkpoint(params, iteration)

        if best_loss < previous_best - min_delta:
            no_improvement = 0
            previous_best = best_loss
        else:
            no_improvement += 1

        if no_improvement >= patience:
            raise EarlyStop()

    stopped_early = False

    try:
        result = minimize(
            fun=lognormal_loss_and_gradient,
            x0=initial_params,
            args=(X_fit, y_fit, lam),
            jac=True,
            method="L-BFGS-B",
            callback=callback,
            options={
                "maxiter": max_iter,
                "ftol": 1e-12,
                "gtol": 1e-8
            }
        )

    except EarlyStop:
        stopped_early = True
        result = None

    history_df = pd.DataFrame(history)

    return {
        "params": best_params,
        "history": history_df,
        "best_iteration": best_iteration,
        "best_subval_log_mse": best_loss,
        "stopped_early": stopped_early,
        "optimizer_result": result
    }


print("Log-normal E5 early-stopping function defined.")


Log-normal E5 early-stopping function defined.


In [6]:

# Section 4.1: Train All 11 Log-Normal E5 Models

import numpy as np
import pandas as pd

lognormal_e5_final_models = {}
lognormal_e5_training_histories = {}
lognormal_e5_training_records = []

for model_name in MODELS:

    print(f"Training Log-Normal E5: {model_name}")

    result = train_lognormal_early_stopping(
        X_fit=X_subtrain_e5,
        y_fit=cost_subtrain[model_name].to_numpy(),
        X_holdout=X_subval_e5,
        y_holdout=cost_subval[model_name].to_numpy(),
        lam=LOGNORMAL_LAMBDA,
        max_iter=200,
        patience=5,
        min_delta=1e-6
    )

    # Retain the best parameter vector
    lognormal_e5_final_models[model_name] = (
        result["params"].copy()
    )

    # Retain the training history
    lognormal_e5_training_histories[model_name] = (
        result["history"].copy()
    )

    # Record training results
    lognormal_e5_training_records.append({
        "model": model_name,
        "lambda": LOGNORMAL_LAMBDA,
        "best_iteration": result["best_iteration"],
        "best_subval_log_mse": result["best_subval_log_mse"],
        "iterations_recorded": len(result["history"]) - 1,
        "stopped_early": result["stopped_early"]
    })

lognormal_e5_training_summary = pd.DataFrame(
    lognormal_e5_training_records
)

print("\nAll 11 Log-Normal E5 models trained.")

display(lognormal_e5_training_summary)


Training Log-Normal E5: WizardLM/WizardLM-13B-V1.2
Training Log-Normal E5: claude-instant-v1
Training Log-Normal E5: claude-v1
Training Log-Normal E5: claude-v2
Training Log-Normal E5: gpt-3.5-turbo-1106
Training Log-Normal E5: gpt-4-1106-preview
Training Log-Normal E5: meta/code-llama-instruct-34b-chat
Training Log-Normal E5: meta/llama-2-70b-chat
Training Log-Normal E5: mistralai/mistral-7b-chat
Training Log-Normal E5: mistralai/mixtral-8x7b-chat
Training Log-Normal E5: zero-one-ai/Yi-34B-Chat

All 11 Log-Normal E5 models trained.


,model,lambda,best_iteration,best_subval_log_mse,iterations_recorded,stopped_early
0,WizardLM/WizardLM-13B-V1.2,0.1,5,0.121749,10,True
1,claude-instant-v1,0.1,5,0.215812,10,True
2,claude-v1,0.1,5,0.185702,10,True
3,claude-v2,0.1,5,0.246660,10,True
4,gpt-3.5-turbo-1106,0.1,5,0.126918,10,True
5,gpt-4-1106-preview,0.1,5,0.300145,10,True
6,meta/code-llama-instruct-34b-chat,0.1,5,0.083031,10,True
7,meta/llama-2-70b-chat,0.1,5,0.105808,10,True
8,mistralai/mistral-7b-chat,0.1,5,0.096132,10,True
9,mistralai/mixtral-8x7b-chat,0.1,5,0.100084,10,True


In [7]:

# Section 5.1: Verify All Log-Normal E5 Checkpoints

import numpy as np
import pandas as pd

verification_records = []

for model_name in MODELS:

    params = lognormal_e5_final_models[model_name]
    history = lognormal_e5_training_histories[model_name]

    # 385 features + intercept = 386 parameters
    assert params.shape == (386,)
    assert np.isfinite(params).all()

    # Predict log-cost on INTERNAL sub-validation only
    predicted_log_cost = (
        params[0] + X_subval_e5 @ params[1:]
    )

    assert np.isfinite(predicted_log_cost).all()

    # Recompute log-MSE
    y_subval = cost_subval[model_name].to_numpy()

    verified_log_mse = np.mean(
        (np.log(y_subval) - predicted_log_cost) ** 2
    )

    recorded_log_mse = history["subval_log_mse"].min()

    np.testing.assert_allclose(
        verified_log_mse,
        recorded_log_mse,
        rtol=1e-8,
        atol=1e-10
    )

    verification_records.append({
        "model": model_name,
        "parameter_count": len(params),
        "verified_subval_log_mse": verified_log_mse,
        "verified": True
    })

lognormal_e5_verification = pd.DataFrame(
    verification_records
)

display(lognormal_e5_verification)

print("\nAll 11 Log-Normal E5 checkpoints verified successfully.")


,model,parameter_count,verified_subval_log_mse,verified
0,WizardLM/WizardLM-13B-V1.2,386,0.121749,True
1,claude-instant-v1,386,0.215812,True
2,claude-v1,386,0.185702,True
3,claude-v2,386,0.246660,True
4,gpt-3.5-turbo-1106,386,0.126918,True
5,gpt-4-1106-preview,386,0.300145,True
6,meta/code-llama-instruct-34b-chat,386,0.083031,True
7,meta/llama-2-70b-chat,386,0.105808,True
8,mistralai/mistral-7b-chat,386,0.096132,True
9,mistralai/mixtral-8x7b-chat,386,0.100084,True



All 11 Log-Normal E5 checkpoints verified successfully.


In [8]:

# Section 6.1: Save Final Log-Normal E5 Models and Training Records

from pathlib import Path
import joblib
import pandas as pd

OUTPUT_DIR = PROJECT_ROOT / "saved_models" / "lognormal_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Combine the training histories of all 11 LLMs
lognormal_e5_training_history = pd.concat(
    [
        history.assign(model=model_name)
        for model_name, history in lognormal_e5_training_histories.items()
    ],
    ignore_index=True
)

# Package trained models with reproducibility metadata
lognormal_e5_bundle = {
    "model_type": "lognormal_log_cost_l2",
    "feature_type": "e5_small_384_plus_scaled_char_length",
    "model_names": list(MODELS),
    "models": lognormal_e5_final_models,
    "lambda": LOGNORMAL_LAMBDA,
    "training_summary": lognormal_e5_training_summary.copy(),
    "feature_count": 385,
    "parameter_count": 386,
    "embedding_model": "intfloat/multilingual-e5-small",
    "embedding_file": (
        "1_data_preparation/data/processed/e5_small/E5_train.npy"
    ),
    "length_preprocessing": preprocessing["length_scaler"],
    "feature_order": [
        "e5_embedding",
        "scaled_char_length"
    ],
    "early_stopping": {
        "patience": 5,
        "min_delta": 1e-6,
        "max_iter": 200,
        "monitor": "subvalidation_log_mse"
    },
    "training_split_file": (
        "saved_models/training_splits/"
        "routerbench_grouped_subtrain_subval_seed42.npz"
    ),
    "prediction_target": "log_inference_cost",
    "prediction_link": "identity_on_log_scale",
    "calibration_status": "not_calibrated",
    "data_usage": "original_training_partition_only"
}

# Define output paths
model_path = (
    OUTPUT_DIR / "lognormal_e5_length_early_stopped.joblib"
)
history_path = (
    OUTPUT_DIR / "lognormal_e5_length_training_history.csv"
)
summary_path = (
    OUTPUT_DIR / "lognormal_e5_length_training_summary.csv"
)

# Save model bundle
joblib.dump(lognormal_e5_bundle, model_path)

# Save training histories and summary
lognormal_e5_training_history.to_csv(
    history_path,
    index=False
)

lognormal_e5_training_summary.to_csv(
    summary_path,
    index=False
)

print("Saved Log-Normal E5 artifacts:\n")

for path in [model_path, history_path, summary_path]:
    print(
        path.name,
        "-",
        path.stat().st_size,
        "bytes"
    )

print("\nAll 11 trained Log-Normal E5 models saved successfully.")


Saved Log-Normal E5 artifacts:

lognormal_e5_length_early_stopped.joblib - 38459 bytes
lognormal_e5_length_training_history.csv - 7659 bytes
lognormal_e5_length_training_summary.csv - 690 bytes

All 11 trained Log-Normal E5 models saved successfully.


In [9]:

# Section 7.1: Reload and Verify Saved Log-Normal E5 Models

import joblib
import numpy as np

# Reload the saved bundle
loaded_bundle = joblib.load(
    OUTPUT_DIR / "lognormal_e5_length_early_stopped.joblib"
)

# Verify saved metadata
assert loaded_bundle["model_type"] == "lognormal_log_cost_l2"
assert loaded_bundle["feature_count"] == 385
assert loaded_bundle["parameter_count"] == 386
assert loaded_bundle["calibration_status"] == "not_calibrated"

assert loaded_bundle["model_names"] == list(MODELS)
assert set(loaded_bundle["models"]) == set(MODELS)

# Verify all 11 parameter vectors and predictions
for model_name in MODELS:

    original_params = lognormal_e5_final_models[model_name]
    loaded_params = loaded_bundle["models"][model_name]

    # Confirm parameters are identical
    np.testing.assert_array_equal(
        original_params,
        loaded_params
    )

    # Predict log-cost using INTERNAL sub-validation only
    original_predictions = (
        original_params[0]
        + X_subval_e5 @ original_params[1:]
    )

    loaded_predictions = (
        loaded_params[0]
        + X_subval_e5 @ loaded_params[1:]
    )

    np.testing.assert_array_equal(
        original_predictions,
        loaded_predictions
    )

print("Saved Log-Normal E5 bundle reloaded successfully.")
print("Models verified:", len(MODELS))
print("Parameters identical: Yes")
print("Predictions identical: Yes")
print("Calibration status:", loaded_bundle["calibration_status"])


Saved Log-Normal E5 bundle reloaded successfully.
Models verified: 11
Parameters identical: Yes
Predictions identical: Yes
Calibration status: not_calibrated


In [10]:

# Create TRAINING_HANDOFF.md directly on your Mac

from pathlib import Path
from textwrap import dedent

PROJECT_ROOT = Path.home() / "Desktop" / "Erdos-Router"
OUTPUT_DIR = PROJECT_ROOT / "saved_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

README_PATH = OUTPUT_DIR / "TRAINING_HANDOFF.md"

readme = dedent("""\
# RouterBench — Regression Model Training Handoff

## 1. Overview

This document describes 44 trained reference cost-prediction
models developed for the Erdős Institute LLM-routing project.

The objective is to predict LLM inference costs from question
features, providing teammates with pretrained models for
subsequent calibration, evaluation, and routing experiments.

Four model families were trained:

| Regression | Features | Models |
|---|---|---:|
| Gamma | Character TF-IDF + length | 11 |
| Log-normal | Character TF-IDF + length | 11 |
| Gamma | E5 embeddings + length | 11 |
| Log-normal | E5 embeddings + length | 11 |
| **Total** | | **44** |

All four pipelines are complete, and their saved model bundles
have been reloaded and verified against the original parameters
and predictions.

## 2. Data Usage — Important

All final training notebooks use only the original RouterBench
training partition containing 21,900 observations.

The training partition was divided into:

- Internal sub-training: 17,520 observations.
- Internal sub-validation: 4,380 observations.

A group-aware split was used, with random_state=42 and
question_group as the grouping variable.

The internal sub-training observations were used to fit coefficients.
The internal sub-validation observations were used exclusively for
early stopping and checkpoint selection.

**The original calibration, validation, and test datasets were
not loaded or used by these final training notebooks.**

Those original datasets are reserved for subsequent team work.

The internal sub-validation set is NOT the original RouterBench
validation set.

The saved split is located at:

    saved_models/training_splits/
        routerbench_grouped_subtrain_subval_seed42.npz

It contains:

    subtrain_idx
    subval_idx

Note: The saved character TF-IDF preprocessing and length scaler
were fitted before this internal split. Therefore, internal
sub-validation estimates should not be treated as fully
preprocessing-independent holdout estimates.

## 3. Feature Representations

### 3.1 Character TF-IDF + Length

The TF-IDF representation consists of:

- 50,000 character TF-IDF features.
- One scaled character-length feature.

Total: 50,001 features.

The original training feature matrix has shape:

    (21900, 50001)

Its files are located under:

    1_data_preparation/data/processed/char_tfidf_50k/

Relevant files:

    X_train.npz
    train_costs.pkl
    train_questions.pkl
    preprocessing.joblib

The preprocessing.joblib dictionary includes:

    tfidf_vectorizer
    length_scaler
    feature_order

For new questions, reuse the saved vectorizer and scaler.
Do not refit them.

### 3.2 E5 Embeddings + Length

The E5 representation consists of:

- 384 E5 embedding features.
- One scaled character-length feature.

Total: 385 features.

The training embeddings are stored at:

    1_data_preparation/data/processed/e5_small/E5_train.npy

The resulting feature matrix has shape:

    (21900, 385)

The recorded embedding model is:

    intfloat/multilingual-e5-small

The same scaled character-length feature used by the TF-IDF
pipeline is appended as the final column.

Important: For new questions, the original E5 encoding pipeline
must be reproduced, including text prefixes, tokenization,
normalization, and preprocessing settings.

The exact embedding-generation procedure should be verified
against the original data-preparation notebook before inference.
Embedding dimensions alone do not guarantee compatibility.

The row order of E5_train.npy must match the corresponding
original training questions.

## 4. Training Methodology

Both regression families used:

| Parameter | Value |
|---|---|
| Optimizer | L-BFGS-B |
| L2 regularization lambda | 0.1 |
| Maximum optimizer iterations | 200 |
| Early-stopping patience | 5 |
| Minimum improvement | 1e-6 |
| Optimizer ftol | 1e-12 |
| Optimizer gtol | 1e-8 |
| Checkpoint | Minimum internal sub-validation loss |

The intercept was not penalized.

For each model, the retained parameter vector contains the
intercept first, followed by the feature coefficients.

Gamma + TF-IDF: 50,002 parameters.
Log-normal + TF-IDF: 50,002 parameters.
Gamma + E5: 386 parameters.
Log-normal + E5: 386 parameters.

The lambda above is the *regression regularization coefficient*,
not a downstream routing cost-quality tradeoff parameter.

### 4.1 Gamma Regression

Gamma regression uses a log link:

    eta = intercept + X @ coefficients
    predicted_mean_cost = exp(eta)

The regularized training objective is:

    0.5 * mean_gamma_deviance
    + 0.5 * lambda * sum(coefficients**2)

Mean Gamma deviance is:

    2 * mean(y / mu - 1 - log(y / mu))

Early stopping monitors unpenalized Gamma deviance on the
internal sub-validation partition.

These are model-based conditional mean-cost predictions.
No additional downstream calibration was performed.

### 4.2 Log-Normal Regression

The log-normal regression predicts log-cost:

    eta = intercept + X @ coefficients

The regularized training objective is:

    0.5 * mean((log(y) - eta)**2)
    + 0.5 * lambda * sum(coefficients**2)

Early stopping monitors unpenalized log-MSE on the internal
sub-validation partition.

Exponentiating eta produces a positive median/geometric-location
cost estimate under the usual log-normal assumptions:

    exp(eta)

This is NOT automatically the conditional expected cost.

A retransformation or calibration correction may be needed before
using the log-normal predictions for expected-cost routing.

The original calibration data has been left untouched for teammates.

## 5. Saved Model Artifacts

All paths below are relative to the Erdős Router project root.

### 5.1 Gamma — TF-IDF + Length

    saved_models/gamma_final/
        gamma_tfidf_length_early_stopped.joblib
        gamma_tfidf_length_training_history.csv
        gamma_tfidf_length_training_summary.csv

### 5.2 Log-Normal — TF-IDF + Length

    saved_models/lognormal_final/
        lognormal_tfidf_length_early_stopped.joblib
        lognormal_tfidf_length_training_history.csv
        lognormal_tfidf_length_training_summary.csv

### 5.3 Gamma — E5 + Length

    saved_models/gamma_final/
        gamma_e5_length_early_stopped.joblib
        gamma_e5_length_training_history.csv
        gamma_e5_length_training_summary.csv

### 5.4 Log-Normal — E5 + Length

    saved_models/lognormal_final/
        lognormal_e5_length_early_stopped.joblib
        lognormal_e5_length_training_history.csv
        lognormal_e5_length_training_summary.csv

The .joblib files contain all 11 parameter vectors for their
respective families, along with training metadata.

The CSV files preserve iteration histories and training summaries.

## 6. Load All Four Model Bundles

Run the following from a trusted local environment with the
required Python dependencies installed.

```python
from pathlib import Path
import joblib

ROOT = Path.home() / "Desktop" / "Erdos-Router"

paths = {
    "gamma_tfidf":
        ROOT / "saved_models/gamma_final/"
        "gamma_tfidf_length_early_stopped.joblib",

    "lognormal_tfidf":
        ROOT / "saved_models/lognormal_final/"
        "lognormal_tfidf_length_early_stopped.joblib",

    "gamma_e5":
        ROOT / "saved_models/gamma_final/"
        "gamma_e5_length_early_stopped.joblib",

    "lognormal_e5":
        ROOT / "saved_models/lognormal_final/"
        "lognormal_e5_length_early_stopped.joblib",
}

bundles = {
    name: joblib.load(path)
    for name, path in paths.items()
}

for name, bundle in bundles.items():
    print(name, len(bundle["models"]))
```

Expected: four families, each containing 11 models.

Security note: Only load joblib files from trusted sources.
joblib loading can execute Python code.

## 7. Generate Predictions from Prepared Features

The following helper accepts a saved bundle and an already
prepared feature matrix.

```python
import numpy as np

def predict_from_bundle(bundle, X):
    expected_features = bundle["feature_count"]

    if X.shape[1] != expected_features:
        raise ValueError("Incorrect feature dimension.")

    predictions = {}

    for model_name in bundle["model_names"]:
        params = bundle["models"][model_name]

        eta = params[0] + X @ params[1:]
        eta = np.asarray(eta).ravel()

        if bundle["model_type"] == "gamma_log_link_l2":
            predictions[model_name] = np.exp(eta)

        elif bundle["model_type"] == "lognormal_log_cost_l2":
            # Preserve the raw log-cost predictions.
            predictions[model_name] = eta

        else:
            raise ValueError("Unknown model type.")

    return predictions
```

For Gamma, outputs estimate mean inference cost.

For log-normal, outputs are on the log-cost scale.
Do not confuse log-cost predictions with expected costs.

The feature matrix must match the corresponding representation:

    TF-IDF + length: 50001 columns
    E5 + length:      385 columns

## 8. Prepare TF-IDF Features for New Questions

Example:

```python
import numpy as np
import pandas as pd
from scipy import sparse

bundle = bundles["gamma_tfidf"]
prep = bundle["preprocessing"]

questions = [
    "Explain the central limit theorem.",
    "Write a Python function for binary search."
]

tfidf = prep["tfidf_vectorizer"].transform(questions)

lengths = pd.DataFrame({
    "char_length": [len(q) for q in questions]
})

scaled_length = prep["length_scaler"].transform(lengths)

X_new_tfidf = sparse.hstack(
    [tfidf, scaled_length],
    format="csr"
)

predictions = predict_from_bundle(
    bundle, X_new_tfidf
)
```

Use the corresponding log-normal TF-IDF bundle to obtain
log-cost predictions for the same feature matrix.

## 9. Prepare E5 Features for New Questions

The original E5 data-preparation pipeline must be reused.

Conceptual steps:

1. Load the original E5 encoder.
2. Apply its exact training-time text formatting.
3. Encode the questions using the same normalization settings.
4. Obtain 384-dimensional embeddings.
5. Transform character lengths with the saved scaler.
6. Concatenate embeddings and scaled lengths in that order.

The Gamma and log-normal E5 bundles contain a
length_preprocessing scaler.

Once compatible E5 embeddings are available:

```python
import numpy as np
import pandas as pd

bundle = bundles["gamma_e5"]

# e5_embeddings must have been generated using the
# original E5 preprocessing and encoding procedure.
# Shape: (number_of_questions, 384)

questions = [
    "Explain the central limit theorem.",
    "Write a Python function for binary search."
]

lengths = pd.DataFrame({
    "char_length": [len(q) for q in questions]
})

scaled_length = bundle[
    "length_preprocessing"
].transform(lengths)

X_new_e5 = np.hstack([
    e5_embeddings,
    scaled_length
])

predictions = predict_from_bundle(
    bundle, X_new_e5
)
```

This is an inference template, not a complete embedding-generation
implementation. Verify the original E5 encoder settings first.

## 10. Reproducibility and Verification

The training notebooks were developed independently for the
four regression/feature combinations.

Each notebook:

- Loads the original training partition.
- Reuses the saved group-aware internal split.
- Defines the training objective and analytical gradient.
- Verifies the gradient numerically.
- Runs L-BFGS-B optimization with early stopping.
- Retains the best internal sub-validation checkpoint.
- Exports all 11 trained models and training histories.
- Reloads and verifies saved model parameters and predictions.

The saved E5 Gamma and E5 log-normal bundles were checked for
identical parameters and predictions following serialization.

The TF-IDF Gamma and log-normal bundles were likewise verified.

These checks establish serialization consistency, not independent
generalization performance.

## 11. Work Reserved for Teammates

The following tasks were intentionally excluded:

- Calibration using the original calibration partition.
- Model selection using the original validation partition.
- Final evaluation using the original test partition.
- Selection of downstream routing tradeoff parameters.
- Final cost-quality routing evaluation.
- Comparison against other routing methods.

Any calibration or selection should use the appropriate reserved
data under the team's agreed experimental protocol.

## 12. Handoff Checklist

- [x] Gamma + TF-IDF models trained and saved.
- [x] Log-normal + TF-IDF models trained and saved.
- [x] Gamma + E5 models trained and saved.
- [x] Log-normal + E5 models trained and saved.
- [x] All four saved bundles reloaded and verified.
- [x] Original calibration, validation, and test partitions unused.
- [ ] Verify exact E5 encoder reproduction from raw text.
- [ ] Test model loading in a fresh teammate environment.
- [ ] Add the final artifacts and documentation to the team repo.

## 13. Important Notes

The four regression families are reference predictors, not
automatically a complete, calibrated routing system.

Gamma and log-normal estimates are not directly equivalent:
one models mean cost, while the other predicts log-cost.

Regularization lambda was fixed at 0.1 for these reference
experiments; it was not selected using the original
validation or calibration partitions.

All training summaries report results from the internal
training-only split.
""")

# Preserve any existing handoff document before replacing it.
if README_PATH.exists():
    backup_path = OUTPUT_DIR / "TRAINING_HANDOFF_PREVIOUS.md"
    backup_path.write_bytes(README_PATH.read_bytes())
    print("Previous document backed up:", backup_path.name)

README_PATH.write_text(readme, encoding="utf-8")

print("\nTRAINING HANDOFF CREATED SUCCESSFULLY")
print("Location:", README_PATH)
print("File size:", README_PATH.stat().st_size, "bytes")
print("Lines:", len(readme.splitlines()))

# Verify the written file
assert README_PATH.exists()
assert "Gamma — E5 + Length" in README_PATH.read_text(
    encoding="utf-8"
)
assert "Log-Normal — E5 + Length" in README_PATH.read_text(
    encoding="utf-8"
)

print("Verification: Passed")



TRAINING HANDOFF CREATED SUCCESSFULLY
Location: /Users/garenchiloyan/Desktop/Erdos-Router/saved_models/TRAINING_HANDOFF.md
File size: 13353 bytes
Lines: 492
Verification: Passed


In [11]:

# Section 1: Independently Verify All Four Model Bundles

from pathlib import Path
import joblib
import numpy as np

ROOT = Path.home() / "Desktop" / "Erdos-Router"

model_paths = {
    "Gamma + TF-IDF":
        ROOT / "saved_models/gamma_final/"
        "gamma_tfidf_length_early_stopped.joblib",

    "Log-normal + TF-IDF":
        ROOT / "saved_models/lognormal_final/"
        "lognormal_tfidf_length_early_stopped.joblib",

    "Gamma + E5":
        ROOT / "saved_models/gamma_final/"
        "gamma_e5_length_early_stopped.joblib",

    "Log-normal + E5":
        ROOT / "saved_models/lognormal_final/"
        "lognormal_e5_length_early_stopped.joblib"
}

bundles = {}

for name, path in model_paths.items():

    assert path.is_file(), f"Missing file: {path}"

    bundle = joblib.load(path)
    bundles[name] = bundle

    assert len(bundle["models"]) == 11

    feature_count = bundle["feature_count"]

    for model_name in bundle["model_names"]:
        params = bundle["models"][model_name]

        assert params.shape == (feature_count + 1,)
        assert np.isfinite(params).all()

    print(
        f"{name}: "
        f"11 models, "
        f"{feature_count} features — VERIFIED"
    )

print("\nAll four model bundles loaded successfully.")
print("Total models:", sum(
    len(bundle["models"])
    for bundle in bundles.values()
))


Gamma + TF-IDF: 11 models, 50001 features — VERIFIED
Log-normal + TF-IDF: 11 models, 50001 features — VERIFIED
Gamma + E5: 11 models, 385 features — VERIFIED
Log-normal + E5: 11 models, 385 features — VERIFIED

All four model bundles loaded successfully.
Total models: 44
